# Graph vs search — локальные модели (LangChain) V2

Повтор eval из [graph-memory-starter](https://github.com/Glitch-Cat-Club/graph-memory-starter): один вопрос, два условия.

- **search** — LangChain-агент сам вызывает инструменты Grep/Read по `corpus-before/`.
- **graph** — SQLite-обход (`src/recall.py`) подставляет факты в промпт; модель **без тулов**.

V2 добавляет в таблицу **время** и **скорость**:

- `elapsed_s` — wall-clock всего условия (у search включая grep/read).
- `ingest_s` / `ingest_tps` — обработка промпта (TTFT / `prompt_eval_duration`), если сервер отдал stats.
- `generate_s` / `generate_tps` — генерация (`generation_time` / `eval_duration` / `tokens_per_second`).
- `overall_tps` — `total_tokens / elapsed_s`.

На OpenAI-совместимом `/v1` (LM Studio) stats часто нет: тогда заполнятся `elapsed_s` и `overall_tps`.

Модель меняется в ячейке **Config**. Нужны Python 3.10+ и локальный сервер (Ollama или LM Studio).

```bash
ollama pull qwen2.5:7b
```

Open from `notebooks/` or the repo root (the directory that contains `src/`).

## 1. Зависимости

In [1]:
%pip install -qU "langchain>=1.0" langchain-ollama langchain-openai langchain-core pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Config — сюда подставляете модель

`PROVIDER = "ollama"` и `MODEL` — то, что меняете между прогонами. Для LM Studio / llama.cpp поставьте `PROVIDER = "openai_compat"` и `OPENAI_BASE`.

In [38]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src" / "recall.py").exists() else HERE.parent
if not (ROOT / "src" / "recall.py").exists():
    raise FileNotFoundError("Open this notebook from notebooks/ or the repo root (the directory that contains src/).")

PROVIDER = "openai_compat"  # "ollama" | "openai_compat"
MODEL = "llama-xlam-2-8b-fc-r-mlx"  # ollama list / имя модели на локальном сервере
OLLAMA_URL = "http://192.168.0.55:1234"
OPENAI_BASE = "http://192.168.0.55:1234/v1"  # LM Studio, llama.cpp server, vLLM
OPENAI_API_KEY = "not-needed"

SEARCH_DIR = ROOT / "corpus-before"  # только unstructured docs
QUESTION = "A customer wants an £800 refund in March. Who signs it off?"
AGENT_RECURSION_LIMIT = 25
RESULTS_CSV = ROOT / "notebooks" / "eval_local_V2_results.csv"

SEARCH_DIR, MODEL, PROVIDER

(PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before'),
 'llama-xlam-2-8b-fc-r-mlx',
 'openai_compat')

## 3. Graph DB и recall (код репозитория, без LLM)

In [39]:
import sys

sys.path.insert(0, str(ROOT / "src"))

from build_graph import main as build_graph
from recall import recall

build_graph()
facts = recall(QUESTION)
memory_text = facts.as_text()
print(memory_text)

built graph.db: 13 entities, 13 relations, 10 aliases from 8 docs
memory: 8 facts recalled in 1 ms

Customer support SOP --[references]--> Refund approvals   (customer-support-sop.md)
Refund approvals --[approved_by]--> Ops Manager           (refund-policy.md)
Customer support SOP --[references]--> Support Lead       (customer-support-sop.md)
Onboarding process --[references]--> Ops Manager          (onboarding-process.md)
Ops Manager --[held_by]--> Sarah Chen                     (org-chart.md)
Sarah Chen --[delegates_to]--> Marcus Webb                (delegation-memo.md)
Incident response --[references]--> Support Lead          (incident-response.md)
Onboarding process --[references]--> Tooling inventory    (onboarding-process.md)

where:
  Customer support SOP: Tickets answered within one business day; refunds under £500 processed by agents; escalation after two replies
  Incident response: Incidents triaged by the Support Lead; severity-1 escalates to the Founder; write-up within 48

## 4. LangChain-модель

`make_chat_model()` — единственная точка смены бэкенда. Search и graph берут один и тот же объект.

In [40]:
from langchain_ollama import ChatOllama


def make_chat_model():
    if PROVIDER == "ollama":
        return ChatOllama(
            model=MODEL,
            temperature=0,
            base_url=OLLAMA_URL,
        )
    if PROVIDER == "openai_compat":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model=MODEL,
            temperature=0,
            base_url=OPENAI_BASE,
            api_key=OPENAI_API_KEY,
        )
    raise ValueError(f"Unknown PROVIDER={PROVIDER!r}. Use 'ollama' or 'openai_compat'.")


llm = make_chat_model()
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x13fdbf290>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x13fdbc510>, root_client=<openai.OpenAI object at 0x1053dce10>, root_async_client=<openai.AsyncOpenAI object at 0x13fdbc390>, model_name='llama-xlam-2-8b-fc-r-mlx', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://192.168.0.55:1234/v1', stream_chunk_timeout=120.0)

## 5. Инструменты search (Grep + Read), корень только `corpus-before/`

In [41]:
import re
from langchain.tools import tool

SEARCH_ROOT = SEARCH_DIR.resolve()


def _safe_md(rel: str) -> Path:
    path = (SEARCH_ROOT / rel).resolve()
    if not path.is_relative_to(SEARCH_ROOT) or path.suffix != ".md" or not path.is_file():
        raise ValueError(f"blocked path: {rel}")
    return path


@tool
def grep_notes(pattern: str) -> str:
    """Search markdown notes with a Python regex (case-insensitive). Returns matching lines."""
    rx = re.compile(pattern, re.I)
    hits = []
    for path in sorted(SEARCH_ROOT.glob("*.md")):
        for i, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if rx.search(line):
                hits.append(f"{path.name}:{i}:{line.strip()}")
    return "\n".join(hits[:80]) or "(no matches)"


@tool
def read_note(filename: str) -> str:
    """Read the full text of one markdown note. Pass only the filename, e.g. team.md."""
    return _safe_md(filename).read_text(encoding="utf-8")


TOOLS = [grep_notes, read_note]
list(SEARCH_ROOT.glob("*.md"))

[PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/ooo-march-sarah.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/incident-log-january.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/it-setup.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/newsletter-march-draft.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/expenses.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/holiday-rota-2026.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/supplier-notes.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/customer-ops-handbook.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/brand-voice.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpu

## 6. Search — агент LangChain (`create_agent`)

In [44]:
import time

from langchain.agents import create_agent
from langchain.messages import AIMessage

SEARCH_SYSTEM = """You answer from company markdown notes using tools.
You must follow this exact retrieval protocol:
1. First call grep_notes with a broad regex based on the question.
2. Use only filenames returned by grep_notes. Never invent, infer, or alter a filename.
3. Call read_note only with an exact filename from grep_notes output.
4. If grep_notes returns no matches, do not call read_note; answer that the notes contain insufficient evidence.
After reading the relevant notes, answer concisely and do not guess."""

search_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SEARCH_SYSTEM,
 )

_t0 = time.perf_counter()
search_result = search_agent.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    {"recursion_limit": AGENT_RECURSION_LIMIT},
)
search_elapsed = time.perf_counter() - _t0


def last_text(messages) -> str:
    for m in reversed(messages):
        if isinstance(m, AIMessage) and (m.content or "") and not m.tool_calls:
            c = m.content
            return c if isinstance(c, str) else str(c)
    return ""


def trace_tools(messages):
    names, reads = [], []
    for m in messages:
        for tc in getattr(m, "tool_calls", None) or []:
            name = tc["name"] if isinstance(tc, dict) else tc.name
            args = tc["args"] if isinstance(tc, dict) else tc.args
            names.append(name)
            if name == "read_note":
                reads.append(args.get("filename", ""))
    return names, sorted(set(r for r in reads if r))


search_answer = last_text(search_result["messages"])
search_tool_names, search_docs = trace_tools(search_result["messages"])
print(search_answer)
print("---")
print("tool calls:", search_tool_names)
print("docs read:", search_docs)
print(f"elapsed_s: {search_elapsed:.3f}")

The notes contain insufficient evidence to determine who signs off on a refund.
---
tool calls: ['grep_notes']
docs read: []
elapsed_s: 18.091


In [43]:
sorted(p.name for p in SEARCH_ROOT.glob("*.md"))

['brand-voice.md',
 'customer-ops-handbook.md',
 'expenses.md',
 'holiday-rota-2026.md',
 'incident-log-january.md',
 'it-setup.md',
 'meeting-notes-2026-02-14.md',
 'newsletter-march-draft.md',
 'ooo-march-sarah.md',
 'refunds-policy-2024-superseded.md',
 'supplier-notes.md',
 'team.md']

## 7. Graph — тот же LLM, без инструментов, факты из recall

In [45]:
import time

from langchain.messages import HumanMessage, SystemMessage

GRAPH_SYSTEM = (
    "Answer only from the memory block in the user message. "
    "Do not use tools. Do not invent facts. Be concise."
)
graph_prompt = f"{memory_text}\n\nQuestion: {QUESTION}"

_t0 = time.perf_counter()
graph_msg = llm.invoke(
    [SystemMessage(content=GRAPH_SYSTEM), HumanMessage(content=graph_prompt)]
)
graph_elapsed = time.perf_counter() - _t0

graph_answer = graph_msg.content if isinstance(graph_msg.content, str) else str(graph_msg.content)
graph_contaminated = bool(getattr(graph_msg, "tool_calls", None))
print(graph_answer)
print("---")
print("tool_calls on graph cell:", getattr(graph_msg, "tool_calls", None))
print(f"elapsed_s: {graph_elapsed:.3f}")

The Ops Manager, Sarah Chen, is on leave from 1-31 March. According to the Refund approvals policy, refunds over £500 need Ops Manager sign-off. Since Sarah Chen is unavailable, Marcus Webb, who has full operational authority during her leave, will sign off the refund.
---
tool_calls on graph cell: []
elapsed_s: 19.572


## 8. Скоринг

- **correct** — в ответе Marcus Webb (или однозначный Marcus) как тот, кто подписывает.
- hops: (1) £500 → Ops Manager, (2) роль у Sarah Chen, (3) март / Marcus.
- graph context read ≈ `len(memory_text) / 4`.
- `elapsed_s` у search включает вызовы инструментов, не только LLM.

In [46]:
import pandas as pd
from langchain.messages import AIMessage

HOP1 = re.compile(r"(500|ops manager|operations manager)", re.I)
HOP2 = re.compile(r"sarah", re.I)
HOP3 = re.compile(r"marcus", re.I)


def hops(text: str) -> int:
    return sum(bool(rx.search(text or "")) for rx in (HOP1, HOP2, HOP3))


def _num(*vals):
    for v in vals:
        if v is None:
            continue
        try:
            x = float(v)
        except (TypeError, ValueError):
            continue
        if x == x and x > 0:
            return x
    return None


def _ns_to_s(v):
    if v is None:
        return None
    x = float(v)
    return x / 1e9 if x > 1e6 else x


def usage_of(msg) -> dict:
    meta = getattr(msg, "usage_metadata", None) or {}
    resp = getattr(msg, "response_metadata", None) or {}
    tu = resp.get("token_usage") or resp.get("usage") or {}
    inp = meta.get("input_tokens") or tu.get("prompt_tokens") or tu.get("input_tokens")
    out = meta.get("output_tokens") or tu.get("completion_tokens") or tu.get("output_tokens")
    tot = meta.get("total_tokens") or tu.get("total_tokens")
    if tot is None and inp is not None and out is not None:
        tot = inp + out
    return {"input": inp or 0, "output": out or 0, "total": tot or 0}


def sum_usages(messages) -> dict:
    acc = {"input": 0, "output": 0, "total": 0, "calls": 0}
    for m in messages:
        if not isinstance(m, AIMessage):
            continue
        u = usage_of(m)
        if u["total"] == 0:
            continue
        acc["input"] += u["input"]
        acc["output"] += u["output"]
        acc["total"] += u["total"]
        acc["calls"] += 1
    return acc


def msg_timings(msg):
    resp = getattr(msg, "response_metadata", None) or {}
    stats = resp.get("stats") or {}
    ttft = _num(
        stats.get("time_to_first_token"),
        stats.get("time_to_first_token_seconds"),
        resp.get("time_to_first_token"),
        _ns_to_s(resp.get("prompt_eval_duration")),
    )
    gen_s = _num(
        stats.get("generation_time"),
        stats.get("generation_time_seconds"),
        resp.get("generation_time"),
        _ns_to_s(resp.get("eval_duration")),
    )
    gen_tps = _num(
        stats.get("tokens_per_second"),
        stats.get("generation_tps"),
        resp.get("tokens_per_second"),
    )
    ingest_tps = _num(
        stats.get("prompt_tokens_per_second"),
        stats.get("prompt_tps"),
        resp.get("prompt_eval_tps"),
    )
    return ttft, gen_s, gen_tps, ingest_tps


def speeds(messages, usage, elapsed):
    ttft_sum = gen_s_sum = 0.0
    gen_tps_w = ingest_tps_w = 0.0
    for m in messages:
        if not isinstance(m, AIMessage):
            continue
        ttft, gen_s, gen_tps, ingest_tps = msg_timings(m)
        u = usage_of(m)
        if u["total"] == 0 and ttft is None and gen_s is None:
            continue
        if ttft:
            ttft_sum += ttft
        if gen_s:
            gen_s_sum += gen_s
        if gen_tps and u["output"]:
            gen_tps_w += gen_tps * u["output"]
        if ingest_tps and u["input"]:
            ingest_tps_w += ingest_tps * u["input"]

    elapsed_s = round(elapsed, 3) if elapsed else None
    ingest_s = ttft_sum or None
    decode_s = gen_s_sum or None

    ingest_tps = None
    if ingest_tps_w and usage["input"]:
        ingest_tps = ingest_tps_w / usage["input"]
    elif ingest_s and usage["input"]:
        ingest_tps = usage["input"] / ingest_s

    gen_tps = None
    if gen_tps_w and usage["output"]:
        gen_tps = gen_tps_w / usage["output"]
    elif decode_s and usage["output"]:
        gen_tps = usage["output"] / decode_s

    overall_tps = None
    if elapsed_s and usage["total"]:
        overall_tps = usage["total"] / elapsed_s

    def r(x):
        return None if x is None else round(x, 2)

    return {
        "elapsed_s": elapsed_s,
        #"ingest_s": r(ingest_s),
        #"generate_s": r(decode_s),
        #"ingest_tps": r(ingest_tps),
        #"generate_tps": r(gen_tps),
        "overall_tps": r(overall_tps),
    }


def row(condition, answer, tool_names, docs, usage, elapsed, messages):
    n = hops(answer)
    return {
        "model": MODEL,
        #"provider": PROVIDER,
        "condition": condition,
        "result": "correct" if HOP3.search(answer or "") else "wrong",
        "hops": f"{n} of 3",
        "tool_calls": len(tool_names),
        "docs_read": len(docs),
        "llm_calls": usage.get("calls", 1),
        "prompt_tokens": usage["input"],
        "completion_tokens": usage["output"],
        "total_tokens": usage["total"],
        **speeds(messages, usage, elapsed),
        "answer": (answer or "").strip().replace("\n", " ")[:240],
    }


if graph_contaminated:
    raise RuntimeError("Graph cell invoked tools — rerun; protocol requires 0 retrieval.")

search_usage = sum_usages(search_result["messages"])
graph_usage = usage_of(graph_msg)
graph_usage["calls"] = 1

table = pd.DataFrame(
    [
        row(
            "search",
            search_answer,
            search_tool_names,
            search_docs,
            search_usage,
            search_elapsed,
            search_result["messages"],
        ),
        row(
            "graph",
            graph_answer,
            [],
            [],
            graph_usage,
            graph_elapsed,
            [graph_msg],
        ),
    ]
)
display(table)

,model,condition,result,hops,tool_calls,docs_read,llm_calls,prompt_tokens,completion_tokens,total_tokens,elapsed_s,overall_tps,answer
0,llama-xlam-2-8b-fc-r-mlx,search,wrong,0 of 3,1,0,2,1544,40,1584,18.091,87.56,The notes contain insufficient evidence to det...
1,llama-xlam-2-8b-fc-r-mlx,graph,correct,3 of 3,0,0,1,415,60,475,19.572,24.27,"The Ops Manager, Sarah Chen, is on leave from ..."


## 9. Запись прогона в CSV

Файл отдельно от V1 (`notebooks/eval_local_V2_results.csv`), чтобы не смешивать схемы колонок. Индекс не пишется.

In [47]:
from datetime import datetime

run = table.copy()
run.insert(0, "run_at", datetime.now().isoformat(timespec="seconds"))

if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    combined = pd.concat([prev, run], ignore_index=True)
else:
    combined = run

combined.to_csv(RESULTS_CSV, index=False)
display(combined)
print("wrote", RESULTS_CSV.resolve())

,run_at,model,condition,result,hops,tool_calls,docs_read,llm_calls,prompt_tokens,completion_tokens,total_tokens,elapsed_s,overall_tps,answer
0,2026-09-04T14:07:24,gpt-oss-20b,search,wrong,1 of 3,2,1,3,2085,230,2315,105.125,22.02,The **Operations Manager** must sign off an £8...
1,2026-09-04T14:07:24,gpt-oss-20b,graph,correct,1 of 3,0,0,1,468,106,574,23.354,24.58,Marcus Webb.
2,2026-09-04T14:14:15,hermes-3-llama-3.1-8b,search,wrong,0 of 3,2,0,3,1760,267,2027,52.789,38.40,Apologies for the confusion. Based on the limi...
3,2026-09-04T14:14:15,hermes-3-llama-3.1-8b,graph,correct,3 of 3,0,0,1,414,41,455,12.405,36.68,"Sarah Chen, the Ops Manager, and Marcus Webb. ..."
4,2026-09-04T14:23:16,google/gemma-4-e2b,search,wrong,1 of 3,2,0,3,1199,901,2100,89.684,23.42,The refund must be signed off by the **Operati...
5,2026-09-04T14:23:16,google/gemma-4-e2b,graph,correct,1 of 3,0,0,1,446,358,804,35.752,22.49,Marcus Webb
6,2026-09-04T14:30:49,llama-xlam-2-8b-fc-r-mlx,search,wrong,0 of 3,1,0,2,1544,40,1584,18.091,87.56,The notes contain insufficient evidence to det...
7,2026-09-04T14:30:49,llama-xlam-2-8b-fc-r-mlx,graph,correct,3 of 3,0,0,1,415,60,475,19.572,24.27,"The Ops Manager, Sarah Chen, is on leave from ..."


wrote /Users/semenoffalex/Agents/Cursor/graph-memory-starter/eval_local_V2_results.csv
